In [1]:
import pyarrow.dataset as ds

dataset = ds.dataset(
    "../output/illustris_skirt.parquet",
    format="parquet",
    exclude_invalid_files=True,
)
print(dataset.count_rows())
dataset.schema

376


image: list<element: list<element: list<element: float>>>
  child 0, element: list<element: list<element: float>>
      child 0, element: list<element: float>
          child 0, element: float
simulation: string
snapshot: int32
subhalo_id: int32
-- schema metadata --
huggingface: '{"info": {"features": {"image": {"feature": {"feature": {"f' + 259

In [4]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

n_cols = 5
n_rows = 4
page_size = n_cols * n_rows
n_total = dataset.count_rows()
n_pages = (n_total + page_size - 1) // page_size

slider = widgets.IntSlider(value=0, min=0, max=n_pages - 1, description="Page:")
out = widgets.Output()


def show_page(page):
    start = page * page_size
    # Read only the current page from disk
    batch = dataset.to_table(columns=["image"]).slice(start, page_size).to_pydict()["image"]
    with out:
        out.clear_output(wait=True)
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(25, 20))
        for ax, img_channels in zip(axes.flatten(), batch):
            data = np.stack([np.stack(ch) for ch in img_channels]).transpose(1, 2, 0) * 255
            image = Image.fromarray(data.astype(np.uint8), "RGB")
            ax.imshow(image)
            ax.axis("off")
        for ax in axes.flatten()[len(batch) :]:
            ax.axis("off")
        plt.tight_layout()
        plt.show()
        plt.close(fig)


slider.observe(lambda change: show_page(change["new"]), names="value")
display(slider, out)
show_page(0)


IntSlider(value=0, description='Page:', max=18)

Output()